#Market Microstructure

The objective of this project is to apply concepts studying from Christopher Bishop's Pattern Recognition and Machine Learning book. 

Let's break down the project in the following parts:

##Modeling Price and Uncertainty using the Gaussian:

Starting from the initial assumption that the distribution of the price is as follows: 

$$p(x) = \mathcal{N}(x | \mu, \sigma^2)$$

The mean is our best estimation of the true price, while the variance represents the volatility, it is the uncertainty around our price estimation.

We also introduce the concept of precision, being the inverse of the variance. It is high when the market is calm and low when the market is high.

##About log returns

Most of the time, log returns are measured instead of prices, making the Gaussian a much better fit:

$$r = \ln(S_{t}/S_{t-1})$$

Furthermore, if log returns are Gaussian, the prices themselves follow a log-normal distribution, which is STRICTLY positive. 

#Bayesian Inference

Using Bayesian inference logic: 

$$\text{Posterior} \propto \text{Likelihood} \times \text{Prior}$$

If both prior and likelihood are Gaussian, then posterior is Gaussian. This is called conjugacy.

We will use this approach to update our best guess ($\mu$), and our certainty ($\lambda$).

The posterior $\mu$ is basically a weighted avergae of the prior and likelihood precisions, so from how much we trust our current knowledge and the new data point. 

$$\mu_{post} = \frac{\lambda_0}{\lambda_0 + \lambda_L}\mu_0 + \frac{\lambda_L}{\lambda_0 + \lambda_L}x$$

So, the expected average doesn't stay the same but moves way less than it would for a stable point. Thereofore, more volatile sudden moves are treated as noise compared to actual more trusted and weighted new data points.

This is exactly what Robbins Monro stochastic approximation algortihm formalizes. It uses a gain factor to decide which step to take towards a better estimate: 

$$\theta_{new} = \theta_{old} + a_n \times (\text{Error})$$

It the gaussian case, the gain is the ratio of the new data's precision to the total precision.

The this framework, the goal is to find the true underlying state by minimizing the error function (which is equivalent to maximizing the likelihood function). 

We will update our estimate using a sequence of step sizes:

$$\theta_{n+1} = \theta_n + \alpha_n [f(x_n) - \text{target}]$$

We, however, want the algorithm to completely disregard outliers, so we need to follow these mathematical rules: 

$$\lim_{n->\infty} \alpha_n = 0$$

$$\sum_{n=1}^{\infty} \alpha_n = \infty$$

$$\sum_{n=1}^{\infty} \alpha_n^2 < \infty$$

This will decrease the step taken over time and get the algorithm more stubborn as it gathers more data, effectively until disregarding anomalies completely. 

However, the limit to that is that if the model becomes too stubborn it will lack adaptation if the price of an asset actually jumps to a new level permanently. Therefore, the way to address this issue is to reduce the steps of noise reduction. 

What can be done is either using a switching strategy or a decaying step size to manage the stability/speed trade-off: 

Firstly, while starting, when we will detect a massive shift in market regime, we will keep $\alpha_n^2$ relatively large. This will allow the model to swing quickly to a new price area. As we get closer to the root (fair price), we will reduce $\alpha_n^2$. It will filter out the high frequency noise and prevent the algorithm from over-leveraging on fake signals.

Moreover, in order to estimate the best parameters for of Robbins-Monro formula will be to use: 

$$\mu_{n} = \mu_{n-1} + \alpha_n (x_n - \mu_{n-1})$$

with $\alpha_n = \frac{\sigma^2}{n}$